In [0]:
# shton 3 metadata columns: source file, ingestion timestamp, batch id

from pyspark.sql import functions as F

def add_ingestion_metadata(df, batch_id):
    return (
        df.withColumn("ingestion_timestamp", F.current_timestamp())
          .withColumn("source_file", F.col("_metadata.file_path"))
          .withColumn("batch_id", F.lit(batch_id))
    )


In [0]:
# ndarje opsionale — vetem per tabelat e medha
def write_to_bronze(df, target_table, batch_id, partition_col=None):
    df = add_ingestion_metadata(df, batch_id)
    writer = df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    if partition_col:
        writer = writer.partitionBy(partition_col)
    writer.saveAsTable(target_table)